# FARUMASI shift dispatcher — T4 sweep

Runtime: **GPU → T4**. This notebook trains DQN, REINFORCE, PPO, and A2C for 100,000 frames each, across 10 hyperparameter settings (40 runs). The environment is the same morning pharmacy shift in every run. A finished run is skipped if you start the cell again, so a disconnect keeps the experiments that already completed.

Opening this notebook from GitHub loads only the notebook. The next cell clones the repository when `main.py` is not already in the working directory. Colab already has a CUDA build of PyTorch. The install line does not replace that wheel.

In [ ]:
!nvidia-smi
import torch
print("cuda", torch.cuda.is_available(), getattr(torch.version, "cuda", None))

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/Emmanuel-NS/Rl_Mission-based_Summative.git"
root = Path.cwd()
candidates = [
    root,
    root / "Rl_Mission-based_Summative",
    root / "emmanuel_nsabagasani_rl_summative",
    Path("/content/Rl_Mission-based_Summative"),
]
for folder in candidates:
    if (folder / "main.py").exists():
        os.chdir(folder)
        break
else:
    dest = Path("/content/Rl_Mission-based_Summative")
    if dest.exists():
        subprocess.check_call(["git", "-C", str(dest), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(dest)])
    os.chdir(dest)
print("working directory", Path.cwd())

In [ ]:
!pip install -q --upgrade-strategy only-if-needed "gymnasium>=1.0" pygame matplotlib "torchrl>=0.6" "tensordict>=0.6"
import torch
assert torch.cuda.is_available(), "CUDA disappeared after install. Restart the runtime and run this cell again without reinstalling torch."

In [ ]:
!python main.py --baselines
!python main.py --screenshot

In [ ]:
# 40 runs. On a T4 this is usually about one to two hours.
!python main.py --sweep all --device cuda --frames 100000 --eval-every 10000

In [ ]:
import shutil
from pathlib import Path
shutil.make_archive("/content/farumasi_rl_results", "zip", ".", ".")
print("Download farumasi_rl_results.zip and keep models/best_agent.pt, logs/, and assets/environment.png.")
try:
    from google.colab import files
    files.download("/content/farumasi_rl_results.zip")
except Exception as error:
    print(error)